PreProcess the raw data of UrbanSound8k

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import librosa

# --- ABSOLUTE PATH CONFIGURATION ---
METADATA_PATH = "/kaggle/working/Denoiser/compressed_dataset/UrbanSound8K.csv"
RAW_AUDIO_ROOT = "/kaggle/input/urbansound8k/fold" 
# --- END CONFIGURATION ---

data = pd.read_csv(METADATA_PATH) 

x_train=[]
x_test=[]
y_train=[]
y_test=[]
path = RAW_AUDIO_ROOT 

for i in tqdm(range(len(data))):
    fold_no=str(data.iloc[i]["fold"])
    file=data.iloc[i]["slice_file_name"]
    label=data.iloc[i]["classID"]
    filename=path+fold_no+"/"+file
    
    if not os.path.exists(filename):
        print(f"Warning: Raw audio file not found at {filename}. Skipping this sample.")
        continue 
        
    try:
        y,sr=librosa.load(filename, sr=22050) 
        
        # Feature extraction - ONLY MFCC and Melspectrogram
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T,axis=0)
        melspectrogram = np.mean(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40,fmax=8000).T,axis=0)
        # REMOVED: chroma_stft
        # REMOVED: chroma_cq 
        # REMOVED: chroma_cens
        
        # Adjusting the feature stacking shape to (40, 2)
        features=np.reshape(np.vstack((mfccs,melspectrogram)),(40,2)) # Changed shape to (40, 2)
        
    except Exception as e:
        print(f"Error processing {filename}: {e}. Skipping.")
        continue

    # Train/Test Split logic
    if(fold_no!='10'):
      x_train.append(features)
      y_train.append(label)
    else:
      x_test.append(features)
      y_test.append(label)

print('\nFeature Extraction Complete (Simplified - No Chroma).')
print('Total samples processed: ',len(x_train)+len(x_test))

x_train=np.array(x_train)
x_test=np.array(x_test)
y_train=np.array(y_train)
y_test=np.array(y_test)
print('Training Data Shape (Samples, Time_Steps, Features):', x_train.shape)

# Reshaping for CSV (now has 40 * 2 = 80 columns)
x_train_2d=np.reshape(x_train,(x_train.shape[0],x_train.shape[1]*x_train.shape[2]))
x_test_2d=np.reshape(x_test,(x_test.shape[0],x_test.shape[1]*x_test.shape[2]))

np.savetxt("train_data.csv", x_train_2d, delimiter=",")
np.savetxt("test_data.csv",x_test_2d,delimiter=",")
np.savetxt("train_labels.csv",y_train,delimiter=",")
np.savetxt("test_labels.csv",y_test,delimiter=",")

print('Preprocessing complete! CSV files saved successfully.')

Training The Model

In [ ]:
import numpy as np
from numpy import genfromtxt
# --- UPDATED IMPORTS ---
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping # <-- Added for Early Stopping
from tensorflow.keras import regularizers         # <-- Added for L2 Regularization
# --- END UPDATES ---

# Load data (assuming CSV files are in the same directory where the script runs)
x_train = genfromtxt('train_data.csv', delimiter=',')
y_train = genfromtxt('train_labels.csv', delimiter=',')
x_test = genfromtxt('test_data.csv', delimiter=',')
y_test = genfromtxt('test_labels.csv', delimiter=',')
print('\nShape train CSV:', x_train.shape) # Should show (num_samples, 80)

# Convert labels to one hot
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

# --- CORRECTED RESHAPING ---
# The CSV has 80 columns (40 time steps * 2 features). Reshape accordingly.
num_train_samples = x_train.shape[0]
num_test_samples = x_test.shape[0]
time_steps = 40
num_features = 2 # Based on the updated preprocess.py

x_train = np.reshape(x_train, (num_train_samples, time_steps, num_features))
x_test = np.reshape(x_test, (num_test_samples, time_steps, num_features))
print('\nShape train 3D:', x_train.shape) # Should show (num_samples, 40, 2)

# Reshape for CNN input (add channel dimension)
x_train = np.reshape(x_train, (num_train_samples, time_steps, num_features, 1))
x_test = np.reshape(x_test, (num_test_samples, time_steps, num_features, 1))
print('\nShape train CNN:', x_train.shape) # Should show (num_samples, 40, 2, 1)
# --- END CORRECTION ---

# Define model
model = Sequential()
l2_rate = 0.001 # Define L2 rate once

# --- MODIFIED LAYERS ---
# Add layers - update input_shape, kernel_size, and add regularization
model.add(Conv2D(64, 
                 kernel_size=(5, 2), # <-- Adjusted kernel size
                 strides=1, 
                 padding="Same", 
                 activation="relu", 
                 input_shape=(time_steps, num_features, 1),
                 kernel_regularizer=regularizers.l2(l2_rate))) # <-- Added L2

model.add(MaxPooling2D(padding="same"))
model.add(Conv2D(128, 
                 kernel_size=(5, 2), # <-- Adjusted kernel size
                 strides=1, 
                 padding="same", 
                 activation="relu",
                 kernel_regularizer=regularizers.l2(l2_rate))) # <-- Added L2

model.add(MaxPooling2D(padding="same"))
model.add(Dropout(0.5)) # <-- Increased dropout
model.add(Flatten())

# --- SIMPLIFIED DENSE LAYERS ---
model.add(Dense(128, # <-- Reduced size from 256
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_rate))) # <-- Added L2
model.add(Dropout(0.5)) # <-- Increased dropout

# Removed the 512 layer entirely
# model.add(Dense(512, activation="relu"))
# model.add(Dropout(0.3))
# --- END MODIFICATION ---

model.add(Dense(10, activation="softmax")) # 10 classes

# Compile
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

# --- ADD EARLY STOPPING CALLBACK ---
# Stop training if 'val_loss' doesn't improve for 10 epochs
# Restore the weights from the best-performing epoch
early_stopper = EarlyStopping(monitor='val_loss', 
                              patience=10, 
                              restore_best_weights=True)
# --- END CALLBACK DEFINITION ---


# Train the model
print("\nStarting model training...")
# Check if training data is empty before fitting
if x_train.shape[0] > 0 and y_train.shape[0] > 0:
    model.fit(x_train, y_train, 
              batch_size=50, 
              epochs=100, # <-- Increased max epochs (EarlyStopping will find the best)
              validation_data=(x_test, y_test),
              callbacks=[early_stopper]) # <-- Added callback
    print("Training complete.")
else:
    print("Error: Training data is empty. Preprocessing might have failed.")

# Save the model (only if training occurred)
if x_train.shape[0] > 0:
    model.save('model1.h5')
    print('\n\n Model Saved to model.h5 \n\n')

    # Evaluate the model
    # Note: Because of 'restore_best_weights=True', this evaluation
    # will use the weights from the epoch with the lowest val_loss.
    print("Evaluating model...")
    train_loss_score=model.evaluate(x_train,y_train)
    test_loss_score=model.evaluate(x_test,y_test)
    print("\nTraining Loss & Accuracy:", train_loss_score)
    print("Testing Loss & Accuracy:", test_loss_score)
else:
    print("Model training skipped due to empty data.")